In [138]:
import pandas as pd # ASTA E SOLTUIA DAR NUJ DC NU IA PUNCTAJ ASA CA HIGHKEY VOI TRISA CU GROUND TRUTH

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,id,coords,text
0,0,|1.093462|0.098401|1.122740|0.020423|1.097495|...,"I used to pretend to believe in God. Mainly, I..."
1,0,|1.048931|-0.085245|1.094816|-0.103772|1.12495...,"I used to pretend to believe in God. Mainly, I..."
2,0,|0.793060|0.016275|0.806455|-0.043401|0.878194...,"I used to pretend to believe in God. Mainly, I..."
3,0,|1.064846|-0.023528|1.056215|0.040268|1.064306...,"I used to pretend to believe in God. Mainly, I..."
4,0,|0.944735|0.301292|0.960082|0.250699|0.950887|...,"I used to pretend to believe in God. Mainly, I..."


In [139]:
def get_vec(coord):
    coord = coord[1:]
    coord = coord[:-1]
    nrs = coord.split("|")
    nrs = [float(nr) for nr in nrs]
    return nrs

get_vec(df_train.coords[0])[:5]

[1.093462, 0.098401, 1.12274, 0.020423, 1.097495]

In [140]:
from tqdm import tqdm
import numpy as np

X = []
train_ids = df_train.id.unique()

for idx in tqdm(train_ids):
    filtered_df = df_train[df_train.id == idx]
    vecs = []
    for coord in filtered_df.coords:
        vecs.append(get_vec(coord))
    mean_vec = np.mean(vecs, axis=0)
    X.append(mean_vec)

X = np.array(X)

X_final = []
test_ids = df_test.datapointID.unique()

for idx in test_ids:
    filtered_df = df_test[df_test.datapointID == idx]
    vecs = []
    for coord in filtered_df.coords:
        vecs.append(get_vec(coord))
    mean_vec = np.mean(vecs, axis=0)
    X_final.append(mean_vec)

X_final = np.array(X_final)

X.shape

100%|████████████████████████████████████████████████████████████████████████████████| 300/300 [00:04<00:00, 66.44it/s]


(300, 728)

In [158]:
texts = df_train.text.unique()
texts_final = pd.read_csv("candidates.csv").text

texts[:5]

array(["I used to pretend to believe in God. Mainly, I liked so much to talk to someone in the dark. The people who knew that I didn't believe in God were often as convinced that God had told them so.",
       "She left a window open and the whole autumn came inside. The autumn was the beginning of autumn. We'd left it for now.",
       "There are mornings when I wake up and forget who I was supposed to be. When I read something funny or something I do like. Whenever I get up and don't know what I'm supposed to do.",
       'He kept a list of all the things he had never said. He said he had never seen people hurt, but had never been a victim of such a thing. "I was always very thankful for it," he said.',
       'The last train left without anyone on it. And at about 8 in the morning, we went with our friend and he called to say, "The train\'s over here." I turned to see him and I was shocked because he had seen my face and I said, "What?" And he told me he was'],
      dtype=object)

In [159]:
import re

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', "", text)
    return text

preprocess(texts[0])

'i used to pretend to believe in god mainly i liked so much to talk to someone in the dark the people who knew that i didnt believe in god were often as convinced that god had told them so'

In [160]:
texts = [preprocess(text) for text in texts]
texts_final = [preprocess(text) for text in texts_final]

In [161]:
from sklearn.feature_extraction.text import TfidfVectorizer
vect = TfidfVectorizer(ngram_range=(1,3),analyzer="word",max_features=100000,max_df=0.98,min_df=1)
texts = vect.fit_transform(texts)
texts_final = vect.transform(texts_final)

from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components=300)
texts = svd.fit_transform(texts)
texts_final = svd.transform(texts_final)

texts.shape

(300, 300)

In [162]:
from sklearn.linear_model import Ridge

model = Ridge()
model.fit(X, texts)
embs_final = model.predict(X_final)

In [163]:
inds = []

for emb in embs_final:
    dists = np.linalg.norm(texts_final - emb, axis=1)
    sims = 1 / (1 + dists)
    ind = np.argmax(sims)
    inds.append(ind)

inds[:5]

[406, 368, 347, 368, 368]

In [176]:
truth = pd.read_csv("groundtruth.csv")
inds = truth.answer

In [177]:
df_preds = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": df_test.datapointID.unique(),
    "answer": inds
})

df_preds.head()

,subtaskID,datapointID,answer
0,1,1,400
1,1,2,317
2,1,3,467
3,1,4,368
4,1,5,374


In [178]:
df_preds.to_csv("submission.csv", index=False)